# Hidden State Extraction — Trajectory

Extracts `h_inst` and `h_post_inst` at the Zhao et al. token positions for all conversations.

```
... [user content] <|eot_id|> <|start_header_id|> assistant <|end_header_id|> \n\n
         ↑               ↑
      t_inst        t_post_inst
```

For each conversation, one forward pass per turn k = 1..n runs at increasing context depth.
This produces the full trajectory. The final-turn representation is the last row (turn_k == n_accepted_turns) — no separate Mode A extraction is needed.

**Output:**
- `results/representations/trajectory/{FOLDER_NAME}/` — full turn-by-turn trajectory
- `results/representations/single_turn/{harmful|benign}/` — single-turn JBB baseline (no attack context)

Each output directory contains:
  - `h_inst.npy` — `(N, 32, 4096)` float16
  - `h_post_inst.npy` — `(N, 32, 4096)` float16
  - `metadata.parquet`


In [ ]:
import os
# ── Set visible GPUs before importing torch ───────────────────────────────────
# Adjust to the physical GPU IDs available on your node.
GPU_IDS = [4, 5, 6]                    # ← physical GPU IDs
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join(str(g) for g in GPU_IDS)
LOGICAL_GPU_IDS = list(range(len(GPU_IDS)))

import json
import sys
import threading
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

repo_root = Path("..").resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

# ── Config ────────────────────────────────────────────────────────────────────
FOLDER_NAME = "crescendo_harmful"      # ← change per run

MODEL_ID  = "meta-llama/Llama-3.1-8B-Instruct"
DTYPE     = torch.bfloat16

# All 32 transformer layers (hidden_states[0] = embedding, skip)
LAYER_INDICES = list(range(1, 33))     # → indices into outputs.hidden_states
N_LAYERS      = len(LAYER_INDICES)     # 32

CONV_DIR  = repo_root / "data" / "conversations" / FOLDER_NAME
REPR_ROOT = repo_root / "results" / "representations"

parts     = FOLDER_NAME.split("_")
FRAMEWORK = parts[0]
SPLIT     = parts[1]

assert CONV_DIR.exists(), f"Conversation folder not found: {CONV_DIR}"
files = sorted(CONV_DIR.glob("*.json"))

print(f"Folder:    {FOLDER_NAME}")
print(f"Framework: {FRAMEWORK}")
print(f"Split:     {SPLIT}")
print(f"Files:     {len(files)}")
print(f"GPUs:      {GPU_IDS} → logical {LOGICAL_GPU_IDS}")
print(f"Layers:    {LAYER_INDICES[0]}..{LAYER_INDICES[-1]}  ({N_LAYERS} total)")

In [ ]:
# ── Messages construction ─────────────────────────────────────────────────────

def build_messages_final_turn(conv, framework):
    """
    Build messages ending at the final user turn (no trailing assistant message).
    Used by the sanity check and to compute n_accepted_turns for trajectory extraction.

    Crescendo: rolled-back turns (is_refusal=True) are excluded.
    System prompt taken from target_system_prompt in the JSON.

    Returns (messages, n_accepted_turns) or (None, 0) if no valid turns.
    """
    turns         = conv.get("turns", [])
    system_prompt = conv.get("target_system_prompt", "")

    by_idx = {}
    for t in turns:
        by_idx.setdefault(t["turn_idx"], {})[t["role"]] = t

    accepted = []
    for turn_idx in sorted(by_idx):
        pair   = by_idx[turn_idx]
        user_t = pair.get("user")
        asst_t = pair.get("assistant")
        if not user_t or not asst_t:
            continue
        # Skip rolled-back turns (is_refusal=True on assistant turn)
        if asst_t.get("is_refusal", False):
            continue
        accepted.append((user_t["content"], asst_t["content"]))

    if not accepted:
        return None, 0

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    # All prior turns: user + assistant
    for user_content, asst_content in accepted[:-1]:
        messages.append({"role": "user",      "content": user_content})
        messages.append({"role": "assistant", "content": asst_content})

    # Final turn: user only — no trailing assistant message
    messages.append({"role": "user", "content": accepted[-1][0]})

    return messages, len(accepted)


def build_messages_at_turn_k(conv, framework, k):
    """
    For Mode B: build messages ending at turn k (1-indexed).
    Same rollback filtering as build_messages_final_turn.
    Returns (messages, actual_turn_index) or (None, 0).
    """
    turns         = conv.get("turns", [])
    system_prompt = conv.get("target_system_prompt", "")

    by_idx = {}
    for t in turns:
        by_idx.setdefault(t["turn_idx"], {})[t["role"]] = t

    accepted = []
    for turn_idx in sorted(by_idx):
        pair   = by_idx[turn_idx]
        user_t = pair.get("user")
        asst_t = pair.get("assistant")
        if not user_t or not asst_t:
            continue
        if asst_t.get("is_refusal", False):
            continue
        accepted.append((user_t["content"], asst_t["content"]))

    if not accepted or k > len(accepted):
        return None, 0

    prefix = accepted[:k]  # first k accepted turns

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    for user_content, asst_content in prefix[:-1]:
        messages.append({"role": "user",      "content": user_content})
        messages.append({"role": "assistant", "content": asst_content})

    messages.append({"role": "user", "content": prefix[-1][0]})

    return messages, k


# ── Token position detection ──────────────────────────────────────────────────

def get_positions(tokenizer, input_ids):
    """
    Llama 3.1 Instruct template with add_generation_prompt=True ends as:
        ... {user content}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n

    With no trailing assistant message, the last <|eot_id|> closes the final
    user turn:
        t_post_inst = last <|eot_id|> position
        t_inst      = t_post_inst - 1  (last content token of user turn)

    input_ids: (1, seq_len) tensor.
    Returns (t_inst, t_post_inst) as ints.
    """
    eot_id      = tokenizer.convert_tokens_to_ids("<|eot_id|>")
    eot_pos     = (input_ids[0] == eot_id).nonzero(as_tuple=True)[0]
    t_post_inst = eot_pos[-1].item()
    t_inst      = t_post_inst - 1
    return t_inst, t_post_inst


# ── Forward pass ──────────────────────────────────────────────────────────────

@torch.no_grad()
def extract_at_positions(model, tokenizer, messages):
    """
    Single forward pass. Returns (h_inst, h_post_inst) as (32, 4096) float16 numpy arrays,
    or (None, None) on error.
    """
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    ).to(model.device)

    t_inst, t_post_inst = get_positions(tokenizer, input_ids)

    with torch.autocast(device_type="cuda", dtype=DTYPE):
        outputs = model(input_ids, output_hidden_states=True)

    h_inst = np.stack([
        outputs.hidden_states[l][0, t_inst, :].cpu().to(torch.float16).numpy()
        for l in LAYER_INDICES
    ])  # (32, 4096) float16

    h_post = np.stack([
        outputs.hidden_states[l][0, t_post_inst, :].cpu().to(torch.float16).numpy()
        for l in LAYER_INDICES
    ])  # (32, 4096) float16

    del outputs
    return h_inst, h_post, t_inst, t_post_inst, input_ids.shape[1]


# ── Save helpers ──────────────────────────────────────────────────────────────

def save_arrays(save_dir, h_inst_list, h_post_list, meta_list):
    save_dir.mkdir(parents=True, exist_ok=True)
    h_inst = np.concatenate(h_inst_list, axis=0).astype(np.float16)
    h_post = np.concatenate(h_post_list, axis=0).astype(np.float16)
    meta   = pd.concat(meta_list, ignore_index=True)
    np.save(str(save_dir / "h_inst.npy"),      h_inst)
    np.save(str(save_dir / "h_post_inst.npy"), h_post)
    meta.to_parquet(save_dir / "metadata.parquet", index=False)
    print(f"  Saved → {save_dir}")
    print(f"  h_inst:      {h_inst.shape}  ({h_inst.nbytes/1e9:.2f} GB)")
    print(f"  h_post_inst: {h_post.shape}  ({h_post.nbytes/1e9:.2f} GB)")
    print(f"  rows:        {len(meta)}")


print("Helpers defined.")

In [ ]:
# ── Sanity check: verify t_inst / t_post_inst on 5 conversations ─────────────
# Run before any large extraction. Expected:
#   t_inst      → last word/subword of the final user message (not whitespace)
#   t_post_inst → '<|eot_id|>'

tok_check = AutoTokenizer.from_pretrained(MODEL_ID)
print(f"Sanity-checking 5 conversations from {FOLDER_NAME}\n")

for fpath in files[:5]:
    conv = json.loads(fpath.read_text())
    messages, n_turns = build_messages_final_turn(conv, FRAMEWORK)
    if messages is None:
        print(f"  {fpath.name}: SKIPPED (no accepted turns)")
        continue

    input_ids = tok_check.apply_chat_template(
        messages, return_tensors="pt", add_generation_prompt=True
    )
    t_inst, t_post_inst = get_positions(tok_check, input_ids)

    tok_inst      = tok_check.decode([input_ids[0][t_inst].item()])
    tok_post_inst = tok_check.decode([input_ids[0][t_post_inst].item()])
    context       = tok_check.decode(input_ids[0][t_inst - 3 : t_post_inst + 2].tolist())

    ok_inst      = bool(tok_inst.strip())
    ok_post_inst = (tok_post_inst == "<|eot_id|>")

    print(f"  {fpath.name}")
    print(f"    seq_len={input_ids.shape[1]}  accepted_turns={n_turns}")
    print(f"    t_inst [{t_inst}]      = {repr(tok_inst):20s}  {'OK' if ok_inst else '*** WARNING: whitespace ***'}")
    print(f"    t_post_inst [{t_post_inst}] = {repr(tok_post_inst):20s}  {'OK' if ok_post_inst else '*** WARNING: not eot_id ***'}")
    print(f"    context: {repr(context)}")
    print()

<!-- Mode A removed — trajectory extraction (Mode B) subsumes it -->

In [ ]:
# Mode A removed.
# Final-turn representations = rows where turn_k == n_accepted_turns in trajectory output.
print("Mode A removed. Use trajectory output (last turn_k row per conversation).")

## Single-turn baseline extraction

Extracts `h_inst` / `h_post_inst` for all 200 raw JBB goals (100 harmful + 100 benign)
with no attack wrapper — direct replication of Zhao et al.'s Latent Guard baseline.

Uses the same system prompt as the multi-turn conversations so that any difference
in representations reflects attack context, not system prompt presence.

```python
messages = [
    {"role": "system", "content": BASE_TARGET_PROMPT},
    {"role": "user",   "content": jbb_goal_text},
]
```

**Output:** `results/representations/single_turn/{harmful|benign}/`


In [ ]:
jbb_ds = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors")

# Same system prompt used by all three frameworks during data generation
SHARED_SYSTEM_PROMPT = (
    "You are a helpful, respectful and honest assistant. Always answer as helpfully "
    "as possible, while being safe.\n\nYour answers should not include any harmful, "
    "unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure "
    "that your responses are socially unbiased and positive in nature.\n\nIf a "
    "question does not make any sense, or is not factually coherent, explain why "
    "instead of answering something not correct. If you don't know the answer to a "
    "question, please don't share false information."
)

def run_single_turn_baseline(goal_type):
    """goal_type: 'harmful' or 'benign'"""
    save_dir = REPR_ROOT / "single_turn" / goal_type
    save_dir.mkdir(parents=True, exist_ok=True)

    goals_df = jbb_ds[goal_type].to_pandas()   # Index, Goal, Behavior, Category

    device = f"cuda:{LOGICAL_GPU_IDS[0]}"
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    mdl = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=DTYPE, device_map={"":device}
    )
    mdl.eval()

    h_inst_list, h_post_list, meta_list = [], [], []

    for _, row in tqdm(goals_df.iterrows(), total=len(goals_df),
                       desc=f"single_turn_{goal_type}", file=sys.stdout):
        # Same system prompt as multi-turn conversations — no attack wrapper
        messages = [
            {"role": "system", "content": SHARED_SYSTEM_PROMPT},
            {"role": "user",   "content": row["Goal"]},
        ]
        h_inst, h_post, t_inst, t_post_inst, seq_len = extract_at_positions(mdl, tok, messages)

        h_inst_list.append(h_inst)
        h_post_list.append(h_post)
        meta_list.append({
            "pair_id":     int(row["Index"]),
            "goal_type":   goal_type,
            "behavior":    row["Behavior"],
            "category":    row["Category"],
            "seq_len":     seq_len,
            "t_inst":      t_inst,
            "t_post_inst": t_post_inst,
        })

    save_arrays(
        save_dir,
        [np.stack(h_inst_list)],
        [np.stack(h_post_list)],
        [pd.DataFrame(meta_list)],
    )


run_single_turn_baseline("harmful")
run_single_turn_baseline("benign")


## Trajectory extraction

For each conversation, runs k = 1..n_accepted_turns forward passes at increasing
context depths. Extracts `h_inst` and `h_post_inst` at each turn k.

- Final-turn representations: filter metadata to `turn_k == n_accepted_turns`
- Used for: centroid construction (Design A/B/C), trajectory analysis (RQ2), probe training

**Sample:** set `TRAJ_REPS` below to control which attempt numbers to include.

**Output:** `results/representations/trajectory/{FOLDER_NAME}/`  
Metadata has `turn_k` column (1-indexed prefix depth).


In [ ]:
# ── Trajectory config ───────────────────────────────────────────────────────
TRAJ_REPS = {1, 2, 3, 4, 5}    # which attempt numbers to include

SAVE_TRAJ = REPR_ROOT / "trajectory" / FOLDER_NAME
SAVE_TRAJ.mkdir(parents=True, exist_ok=True)

# Resume support
meta_path_b = SAVE_TRAJ / "metadata.parquet"
if meta_path_b.exists():
    _existing_meta_b = pd.read_parquet(meta_path_b)
    done_b = set(zip(
        _existing_meta_b["pair_id"],
        _existing_meta_b["attempt"],
        _existing_meta_b["turn_k"],
    ))
    existing_h_inst_b = [np.load(str(SAVE_TRAJ / "h_inst.npy"))]
    existing_h_post_b = [np.load(str(SAVE_TRAJ / "h_post_inst.npy"))]
    existing_meta_b   = [_existing_meta_b]
    print(f"Resuming: {len(done_b)} (pair_id, attempt, turn_k) triples already done")
else:
    done_b = set()
    existing_h_inst_b = existing_h_post_b = existing_meta_b = []

sample_files = [
    f for f in files
    if json.loads(f.read_text()).get("attempt", 1) in TRAJ_REPS
]
print(f"Mode B sample: {len(sample_files)} conversations  (reps {sorted(TRAJ_REPS)})", flush=True)

n_gpus    = len(LOGICAL_GPU_IDS)
chunks    = [sample_files[i::n_gpus] for i in range(n_gpus)]
all_h_inst_b = list(existing_h_inst_b)
all_h_post_b = list(existing_h_post_b)
all_meta_b   = list(existing_meta_b)

# Count total (conv, turn) pairs for progress bar
total_pairs = sum(
    len([t for t in json.loads(f.read_text()).get("turns", [])
         if t["role"] == "assistant" and not t.get("is_refusal", False)])
    for f in sample_files
)
pbar = tqdm(total=total_pairs, desc="Trajectory", file=sys.stdout, dynamic_ncols=True)
lock = threading.Lock()

def worker_b(gpu_id, chunk):
    torch.cuda.set_device(gpu_id)
    device = f"cuda:{gpu_id}"
    print(f"GPU {gpu_id}: loading model...", flush=True)
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    mdl = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=DTYPE, device_map={"":device}
    )
    mdl.eval()
    print(f"GPU {gpu_id}: ready, {len(chunk)} files", flush=True)

    batch_h_inst, batch_h_post, batch_meta = [], [], []
    for fpath in chunk:
        conv    = json.loads(fpath.read_text())
        pair_id = conv["objective_pair_id"]
        attempt = conv.get("attempt", 1)

        # Determine how many accepted turns this conversation has
        _, n_accepted = build_messages_final_turn(conv, FRAMEWORK)
        if n_accepted == 0:
            continue

        for k in range(1, n_accepted + 1):
            if (pair_id, attempt, k) in done_b:
                with lock: pbar.update(1)
                continue

            messages, _ = build_messages_at_turn_k(conv, FRAMEWORK, k)
            if messages is None:
                with lock: pbar.update(1)
                continue

            h_inst, h_post, t_inst, t_post_inst, seq_len = extract_at_positions(mdl, tok, messages)

            batch_h_inst.append(h_inst)
            batch_h_post.append(h_post)
            batch_meta.append({
                "conversation_id":    conv.get("conversation_id", fpath.stem),
                "pair_id":            pair_id,
                "goal_type":          conv.get("goal_type", SPLIT),
                "framework":          FRAMEWORK,
                "attempt":            attempt,
                "attack_success":     bool(conv.get("attack_success", False)),
                "n_accepted_turns":   n_accepted,
                "turn_k":             k,
                "seq_len":            seq_len,
                "t_inst":             t_inst,
                "t_post_inst":        t_post_inst,
                "fname":              fpath.name,
            })
            with lock: pbar.update(1)

    with lock:
        if batch_h_inst:
            all_h_inst_b.append(np.stack(batch_h_inst))
            all_h_post_b.append(np.stack(batch_h_post))
            all_meta_b.append(pd.DataFrame(batch_meta))

threads = [threading.Thread(target=worker_b, args=(g, c))
           for g, c in zip(LOGICAL_GPU_IDS, chunks)]
for t in threads: t.start()
for t in threads: t.join()
pbar.close()

save_arrays(SAVE_TRAJ, all_h_inst_b, all_h_post_b, all_meta_b)